# Liver: exact mouse-label permutation test

Refit the fixed ROI-mean GraphSAGE ensemble for every allocation of four
infected labels to eight mice. Each allocation uses eight held-out-mouse
folds and five seeds: **70 × 8 × 5 = 2,800 independently fitted models**.
The cohort-defined, label-free partition is kept fixed.

**Runtime:** many hours to days; benchmark one completed allocation before
estimating. This is separate from the main analysis and is unlikely to
finish in one ordinary Colab session. Use a persistent result folder to
resume. Each allocation/fold/seed checkpoint verifies its labels, graph,
configuration, code and library versions. Learned weights are never reused
across allocations.

The default is the deposited partition for the original conditional test.
Choose `fresh` explicitly after running the [main notebook](mantpy_liver_full.ipynb)
to test its new partition. GPU is unnecessary for the current CPU GNNs.

In [ ]:
from pathlib import Path
import hashlib
from urllib.request import urlopen

_SETUP = Path("experiment_setup.py")
_SETUP_SHA256 = "6faab2e74f2559b2cc76cbb5b0323215a58e15ab2fb355050c55822de4214175"
if not _SETUP.exists():
    with urlopen("https://raw.githubusercontent.com/moeghaf/mantpy_reproducibility/mantpy-api-v1.1.0/notebooks/experiments/experiment_setup.py", timeout=90) as response:
        _setup_bytes = response.read()
    if hashlib.sha256(_setup_bytes).hexdigest() != _SETUP_SHA256:
        raise RuntimeError("Downloaded setup checksum does not match this notebook.")
    _setup_temp = _SETUP.with_name(_SETUP.name + ".download")
    _setup_temp.write_bytes(_setup_bytes)
    _setup_temp.replace(_SETUP)
if hashlib.sha256(_SETUP.read_bytes()).hexdigest() != _SETUP_SHA256:
    raise RuntimeError("Use the setup file matching this notebook release.")
from experiment_setup import prepare_experiment
ASSETS = prepare_experiment("liver")


In [ ]:
import os
from pathlib import Path
import time

import mantpy as mt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from liver_analysis import data, descriptive
import liver_helpers as lh

torch.set_num_threads(min(4, os.cpu_count() or 1))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SELECTED = lh.read_selected()
SEEDS = SELECTED["seeds"]                 # five training seeds: 0–4
RESULTS = Path("results/liver")
RESULTS.mkdir(parents=True, exist_ok=True)
TIMINGS = {}
started = time.perf_counter()
print(f"CNN: {DEVICE}; GNN: CPU; {len(SEEDS)} training seeds")

In [ ]:
PARTITION_MODE = "paper"                # explicitly "paper" or the main run's "fresh"
PERMUTATION_RESULTS = Path("results/liver_permutation") / PARTITION_MODE
PERMUTATION_RESULTS.mkdir(parents=True, exist_ok=True)
# In Colab, set this to a mounted Drive folder to retain model checkpoints.
ALLOCATIONS = None                      # None = all 70; [0] = timing benchmark only
if sorted(SEEDS) != [0, 1, 2, 3, 4]:
    raise ValueError("Use the five distinct paper seeds: 0–4.")
if ALLOCATIONS is not None:
    if not ALLOCATIONS or any(type(i) is not int or not 0 <= i < 70 for i in ALLOCATIONS):
        raise ValueError("Choose allocation indices from 0 to 69.")
    if len(ALLOCATIONS) != len(set(ALLOCATIONS)):
        raise ValueError("Allocation indices must be distinct.")

In [ ]:
stage = time.perf_counter()
liver = mt.datasets.schistosoma_ecm(
    cache_dir=os.environ.get("MANTPY_CACHE"), progressbar=True, verbose=False
)
images, meta = data.cohort(liver)
assert len(images) == 24 and meta.mouse.nunique() == 8
TIMINGS["load_data"] = time.perf_counter() - stage
display(data.mouse_table(meta))

In [ ]:
import json

partition_dir = RESULTS / PARTITION_MODE
if PARTITION_MODE == "paper" and not (partition_dir / "partition.json").exists():
    # The deposited labels need raw coordinates and marker summaries, not a new CNN fit.
    stage = time.perf_counter()
    norm, bounds = data.normalise(images)
    nodes = data.patch_features(norm, meta)
    nodes["signal"] = data.signal_mask(nodes)
    nodes = lh.use_partition(nodes, None, mode="paper")
    TIMINGS["paper_partition_preparation"] = time.perf_counter() - stage
else:
    record = json.loads((partition_dir / "partition.json").read_text())
    for name, field in [("nodes.csv.gz", "nodes_sha256"), ("metadata.csv", "metadata_sha256")]:
        if lh.file_hash(partition_dir / name) != record[field]:
            raise ValueError(f"Changed main-experiment input: {name}")
    nodes = pd.read_csv(partition_dir / "nodes.csv.gz")
    meta = pd.read_csv(partition_dir / "metadata.csv")
print(f"Partition: {PARTITION_MODE}; {int(nodes.signal.sum()):,} signal patches")

In [ ]:
_, graphs = lh.make_graphs(nodes, meta, graph_type="delaunay")
print("Fixed ROI-mean settings, five seeds:")
display(pd.Series(SELECTED["fixed_gnn"]))

## Refit the exact allocation inventory

For each allocation, training labels, feature scaling and class weights
are rebuilt inside each held-out-mouse fold. Seed predictions are averaged
within ROI, then across a mouse's ROIs. The upper-tail statistic is
mouse ROC-AUC, with the observed allocation and ties included.
The genotype-preserving test uses the 36 allocations with two infected
labels within each genotype. Partial runs are timing results, not an exact
P value.

In [ ]:
# Allocation/seed orchestration remains local; every refit uses the public node trainer.
stage = time.perf_counter()
allocations, summary = lh.permutation_test(
    graphs, meta, seeds=SEEDS, config=SELECTED["fixed_gnn"],
    output_dir=PERMUTATION_RESULTS, allocations=ALLOCATIONS
)
TIMINGS["permutation"] = time.perf_counter() - stage
display(allocations)
display(pd.Series(summary, name="fresh exact-test summary"))

In [ ]:
plt.hist(allocations.AUC, bins=np.linspace(0, 1, 17), color="#466A91")
if allocations.observed.any():
    observed_auc = allocations.loc[allocations.observed, "AUC"].iloc[0]
    plt.axvline(observed_auc, color="#BB4A3B", label="Observed allocation")
    plt.legend()
plt.xlabel("Held-out mouse ROC-AUC")
plt.ylabel("Label allocations completed")
plt.show()
if not summary["exact_test_complete"]:
    seconds_per_allocation = TIMINGS["permutation"] / len(allocations)
    print(f"Measured mean: {seconds_per_allocation / 60:.1f} min per completed allocation")
    print(f"70-allocation extrapolation: {70 * seconds_per_allocation / 3600:.1f} h; hardware-specific")

## Labelled deposited references

The deposited paper result is P=4/70 across all allocations and P=2/36
within genotype. Compare the independently refitted result above to these
references after all 70 allocations finish.

In [ ]:
display(pd.DataFrame([
    {"test": "All 70 allocations", "paper_reference": SELECTED["paper_reference"]["permutation_p"]},
    {"test": "36 genotype-preserving allocations", "paper_reference": SELECTED["paper_reference"]["genotype_permutation_p"]},
]))
TIMINGS["notebook_total"] = time.perf_counter() - started
lh.write_json(PERMUTATION_RESULTS / "run.json", {
    "partition_mode": PARTITION_MODE, "config": SELECTED,
    "library_versions": lh.environment(), "timings_seconds": TIMINGS,
    "requested_allocations": "all 70" if ALLOCATIONS is None else ALLOCATIONS,
})
print(f"Results: {PERMUTATION_RESULTS.resolve()}")